# Hope ML: Cloud Training Pipeline (Colab)

This notebook trains the Canonical Causal Transformer model with TS2Vec pre-training and exports the ONNX artifact.
Requires `ticks.csv` to be uploaded or mounted from Google Drive.

## Setup & Configuration

### 1. Colab Secrets Setup
To secure your environment, set the following secrets in the Google Colab "Secrets" tab (key icon in the left sidebar):
- `DERIV_TOKEN`: Your Deriv API token.
- `MODEL_SIGNING_KEY`: The hex-encoded Ed25519 private key for model signing.

**Important**: Enable the "Notebook access" toggle for these secrets in the Colab UI to allow `google.colab.userdata` access.

In [ ]:
from google.colab import drive, userdata
import os
import time
import subprocess

# 1. Mount Google Drive
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

REPO_URL = "https://github.com/planetazul3/hope.git"
TARGET_DIR = "/content/drive/MyDrive/hope"

def run_with_backoff(cmd, max_retries=5):
    for i in range(max_retries):
        try:
            subprocess.check_call(cmd, shell=True)
            return True
        except subprocess.CalledProcessError:
            delay = 2 ** i
            print(f"Command failed. Retrying in {delay}s...")
            time.sleep(delay)
    return False

if not os.path.exists(TARGET_DIR):
    print(f"Cloning {REPO_URL} into Google Drive...")
    # Note: Using depth=1 for faster clone in GDrive
    success = run_with_backoff(f"git clone --depth 1 {REPO_URL} {TARGET_DIR}")
else:
    print(f"Pulling latest changes in {TARGET_DIR}...")
    success = run_with_backoff(f"git -C {TARGET_DIR} pull")

if success:
    print("Repository ready in Google Drive.")
    import sys
    scripts_path = os.path.join(TARGET_DIR, "scripts")
    if scripts_path not in sys.path:
        sys.path.append(scripts_path)
else:
    print("Failed to prepare repository.")

# Export secrets to environment variables
try:
    os.environ["MODEL_SIGNING_KEY"] = userdata.get("MODEL_SIGNING_KEY")
    os.environ["DERIV_TOKEN"] = userdata.get("DERIV_TOKEN")
    os.environ["DERIV_APP_ID"] = "1089" # Default
    print("Secrets loaded into environment.")
except userdata.SecretNotFoundError:
    print("Warning: Secrets not found in Colab userdata. Ensure they are set in the Secrets tab.")


In [1]:
import os
import train_fixed

# Use local repository scripts
train_fixed.main()


Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 2.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 131.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 108.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 133.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 141.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 100.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 116.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 3.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 6.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 5.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━

ModuleNotFoundError: No module named 'train_fixed'